<a href="https://colab.research.google.com/github/ibmm-unibe-ch/FrankenMSA/blob/ngrok/app/FrankenMSA_app_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://github.com/ibmm-unibe-ch/FrankenMSA/blob/dev/app/assets/frankenmsa_header.png?raw=true)

# [FrankenMSA](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/)
This notebook can run the [FrankenMSA App](https://github.com/ibmm-unibe-ch/FrankenMSA/tree/main/) to provide a graphical user interface for manipulating Multiple Sequence Alignments.

We highly recommend to use an external window for best experience. Firefox users might need to use the [ngrok](https://ngrok.com/) option. Otherwise, the inline option might be used as a last option.

Press the "Runtime" --> "Run all" once you made your selection or press "Ctrl + F9".

In [1]:
WindowMode = 'ngrok' # @param ["external", "inline", "ngrok"] {allow-input: true}


# Code

In [ ]:
# === Cell 1: Hard reset + fresh clone of the RIGHT branch ===
import os, sys, shutil, subprocess, json, textwrap, time


try:
    from pyngrok import ngrok
    for t in ngrok.get_tunnels():
        try:
            ngrok.disconnect(t.public_url)
        except:
            pass
    ngrok.kill()
except Exception:
    pass

!pkill -f "python app/app.py" 2>/dev/null || true
!pkill -f "gunicorn"          2>/dev/null || true


shutil.rmtree("/content/FrankenMSA", ignore_errors=True)


!git clone -q --single-branch -b feature/colab-runner https://github.com/ibmm-unibe-ch/FrankenMSA.git /content/FrankenMSA


!git -C /content/FrankenMSA rev-parse --abbrev-ref HEAD
!git -C /content/FrankenMSA log -1 --pretty=oneline


try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "dash==2.16.1", "dash-bootstrap-components==1.5.0",
                           "plotly==5.24.1", "dash-bio==1.0.2", "pyngrok"])
    print("✅ base deps installed")
except Exception as e:
    print("⚠️ deps warning:", e)


try:
    import pathlib, subprocess
    commit = subprocess.check_output(
        ["git","-C","/content/FrankenMSA","rev-parse","--short","HEAD"],
        text=True
    ).strip()
    p = pathlib.Path("/content/FrankenMSA/app/pages/inversefold.py")
    s = p.read_text(encoding="utf-8")
    if "INVERSEFOLD_BUILD_ID" not in s:
        s = s.replace(
            "def layout():",
            f"INVERSEFOLD_BUILD_ID = 'IF-build:{commit}'\n\ndef layout():"
        ).replace(
            'html.H1("Inverse Fold with ProteinMPNN")',
            'html.H1("Inverse Fold with ProteinMPNN"), html.Small(INVERSEFOLD_BUILD_ID, style={"marginLeft":"8px","opacity":0.6})'
        )
        p.write_text(s, encoding="utf-8")
        print("🧩 injected build banner:", commit)
    else:
        print("ℹ️ build banner already present")
except Exception as e:
    print("⚠️ build banner injection skipped:", e)

print("✅ Cell 1 done. Go to Cell 2.")

In [ ]:
# === Cell 2: Start app via ngrok, reusing or creating tunnel ===
import os, sys, time, re
from pathlib import Path
from pyngrok import ngrok, conf

ROOT = "/content/FrankenMSA"
assert os.path.isdir(ROOT), "FrankenMSA repo not found. Run Cell 1 first."


import getpass
PORT = 8050
token = getpass.getpass("Enter ngrok authtoken (hidden): ").strip().strip("'").strip('"')
conf.get_default().auth_token = token

public_url = None
try:
    
    for t in ngrok.get_tunnels():
        addr = (t.config or {}).get("addr","")
        if addr.endswith(f":{PORT}"):
            public_url = t.public_url
            print("♻️ Reusing existing tunnel:", public_url)
            break
    
    if not public_url:
        tun = ngrok.connect(addr=f"0.0.0.0:{PORT}", proto="http")
        public_url = tun.public_url
        print("✅ Created new tunnel:", public_url)
except Exception as e:
    msg = str(e)
    m = re.search(r"https?://[a-z0-9\-]+\.ngrok-[\w\-]+\.(?:dev|app)", msg)
    if m:
        public_url = m.group(0)
        print("♻️ Using tunnel from error message:", public_url)
    else:
        raise


patch_total = 0
for p in Path(f"{ROOT}/app").rglob("*.py"):
    s = p.read_text(encoding="utf-8", errors="ignore")
    s2, n1 = re.subn(r",\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}", ", prevent_initial_call=True", s)
    s3, n2 = re.subn(r"clientside_callback\((.*?)\s*,\s*\{\s*['\"]prevent_initial_call['\"]\s*:\s*True\s*\}\s*\)",
                     r"clientside_callback(\1, prevent_initial_call=True)", s2, flags=re.DOTALL)
    if n1 or n2:
        p.write_text(s3, encoding="utf-8")
        patch_total += n1 + n2
print(f"🩹 Patched {patch_total} place(s)")


import subprocess
env = os.environ.copy()
env["PORT"], env["HOST"] = str(PORT), "0.0.0.0"

proc = subprocess.Popen(
    [sys.executable, "app/app.py"],
    cwd=ROOT,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)


start = time.time()
lines = []
while time.time() - start < 25:
    ln = proc.stdout.readline()
    if ln:
        lines.append(ln.rstrip())
        if "Running on" in ln or "Dash is running" in ln:
            break
    else:
        time.sleep(0.2)

print("\n---- recent logs ----")
print("\n".join(lines[-20:]))
print("---------------------")
print("🌐 Open:", public_url)
print("✅ Look for the banner 'IF-build:xxxx' on the page header to confirm version")